In [1]:
UPDATE = True

In [2]:
PresentDate = "2021-10-08"

In [3]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

SHOW_GRAPHS = True

# Preparation of Data Frames

In [4]:
df  = pd.DataFrame()

## Sentiment

In [5]:
# Reading the saved data pickle file

df_sentiment  = pd.DataFrame()

df_sentiment=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Sentiment.pkl', compression=None)
# df_sentiment.drop(columns=["compound", "neg", "pos", "neu", "Articles", "Title", "Text", "Company"], inplace=True)
df_sentiment.drop(columns=["Company"], inplace=True)
df_sentiment.index = pd.to_datetime(df_sentiment.index)
df_sentiment.Date = pd.to_datetime(df_sentiment.Date)

df_sentiment = df_sentiment.ffill()

df_sentiment["sentiment(t-1)"] = df_sentiment["sentiment"].shift(periods=1)
df_sentiment["sentiment(t-2)"] = df_sentiment["sentiment"].shift(periods=2)
df_sentiment["sentiment(t-1)"] = df_sentiment["sentiment(t-1)"].ffill()
df_sentiment["sentiment(t-2)"] = df_sentiment["sentiment(t-2)"].ffill()

# SENTIMENT RETURN
df_sentiment["sentiment_return"]=(df_sentiment["sentiment(t-1)"]-df_sentiment["sentiment(t-2)"])/df_sentiment["sentiment(t-2)"]
# SENTIMENT LN RETURN
df_sentiment["sentiment_ln_return"]=np.log(df_sentiment["sentiment(t-1)"]/df_sentiment["sentiment(t-2)"])
# SENTIMENT EWM
df_sentiment["sentiment_ewm10"]=df_sentiment["sentiment"].ewm(10).mean()
df_sentiment["sentiment_ewm20"]=df_sentiment["sentiment"].ewm(20).mean()

# POSITIVE LN RETURN
df_sentiment["pos(t-1)"] = df_sentiment["pos"].shift(periods=1)
df_sentiment["pos(t-2)"] = df_sentiment["pos"].shift(periods=2)
df_sentiment["pos(t-1)"] = df_sentiment["pos(t-1)"].ffill()
df_sentiment["pos(t-2)"] = df_sentiment["pos(t-2)"].ffill()
df_sentiment["pos_ln_return"]=np.log(df_sentiment["pos(t-1)"]/df_sentiment["pos(t-2)"])

# NEGATIVE LN RETURN
df_sentiment["neg(t-1)"] = df_sentiment["neg"].shift(periods=1)
df_sentiment["neg(t-2)"] = df_sentiment["neg"].shift(periods=2)
df_sentiment["neg(t-1)"] = df_sentiment["neg(t-1)"].ffill()
df_sentiment["neg(t-2)"] = df_sentiment["neg(t-2)"].ffill()
df_sentiment["neg_ln_return"]=np.log(df_sentiment["neg(t-1)"]/df_sentiment["neg(t-2)"])

# NEUTRAL LN RETURN
df_sentiment["neu(t-1)"] = df_sentiment["neu"].shift(periods=1)
df_sentiment["neu(t-2)"] = df_sentiment["neu"].shift(periods=2)
df_sentiment["neu(t-1)"] = df_sentiment["neu(t-1)"].ffill()
df_sentiment["neu(t-2)"] = df_sentiment["neu(t-2)"].ffill()
df_sentiment["neu_ln_return"]=np.log(df_sentiment["neu(t-1)"]/df_sentiment["neu(t-2)"])

# COMPOUND LN RETURN
df_sentiment["compound(t-1)"] = df_sentiment["compound"].shift(periods=1)
df_sentiment["compound(t-2)"] = df_sentiment["compound"].shift(periods=2)
df_sentiment["compound(t-1)"] = df_sentiment["compound(t-1)"].ffill()
df_sentiment["compound(t-2)"] = df_sentiment["compound(t-2)"].ffill()
df_sentiment["compound_ln_return"]=np.log(df_sentiment["compound(t-1)"]/df_sentiment["compound(t-2)"])

# PRESS RELEASE COUNT
df_sentiment["PressReleaseCount(t-1)"] = df_sentiment["PressReleaseCount"].shift(periods=1)
df_sentiment["PressReleaseCount(t-2)"] = df_sentiment["PressReleaseCount"].shift(periods=2)
df_sentiment["PressReleaseCount(t-1)"] = df_sentiment["PressReleaseCount(t-1)"].ffill()
df_sentiment["PressReleaseCount(t-2)"] = df_sentiment["PressReleaseCount(t-2)"].ffill()
df_sentiment["PressReleaseCount_ln_return"]=np.log(df_sentiment["PressReleaseCount(t-1)"]/df_sentiment["PressReleaseCount(t-2)"])


In [15]:
df_sentiment.loc[df_sentiment["Date"]>"2020-09-20"].head(50).sort_index()#(by=["Date"])

,compound,neg,neu,pos,sentiment,polarity,subjectivity,polarityB,Date,Articles,...,neg_ln_return,neu(t-1),neu(t-2),neu_ln_return,compound(t-1),compound(t-2),compound_ln_return,PressReleaseCount(t-1),PressReleaseCount(t-2),PressReleaseCount_ln_return
2020-09-21,0.9989,0.071,0.812,0.118,62.433862,0.092259,0.427045,Positive,2020-09-21,. WALMART SAYS TARGETS ZERO EMISSIONS BY 2040....,...,1.070441,0.867,0.866,0.001154,0.9825,0.9989,-0.016554,11.0,27.0,-0.897942
2020-09-22,0.9974,0.029,0.888,0.083,74.107143,0.023592,0.304920,Positive,2020-09-22,. DISNEY PARKS SAY HONG KONG DISNEYLAND WILL O...,...,0.707332,0.812,0.867,-0.065539,0.9989,0.9825,0.016554,56.0,11.0,1.627456
2020-09-23,0.9990,0.020,0.843,0.136,87.179487,0.085245,0.411495,Positive,2020-09-23,. HUAWEI SAYS STILL WILLING TO BUY CHIPS FROM ...,...,-0.895384,0.888,0.812,0.089471,0.9974,0.9989,-0.001503,30.0,56.0,-0.624154
2020-09-24,0.9995,0.019,0.857,0.124,86.713287,0.063738,0.367166,Positive,2020-09-24,. GILEAD SCIENCES PRICES $7.25 BLN OF SENIOR U...,...,-0.371564,0.843,0.888,-0.052005,0.9990,0.9974,0.001603,24.0,30.0,-0.223144
2020-09-25,0.9965,0.022,0.854,0.123,84.827586,0.059172,0.454229,Positive,2020-09-25,. JYSELECA® (FILGOTINIB) APPROVED IN JAPAN FOR...,...,-0.051293,0.857,0.843,0.016471,0.9995,0.9990,0.000500,30.0,24.0,0.223144
2020-09-28,0.9990,0.027,0.800,0.173,86.500000,0.039394,0.496994,Positive,2020-09-28,. METLIFE TO REDUCE EMISSIONS BY 30 PERCENT AN...,...,0.146603,0.854,0.857,-0.003507,0.9965,0.9995,-0.003006,14.0,30.0,-0.762140
2020-09-29,0.9993,0.030,0.816,0.154,83.695652,0.058837,0.378926,Positive,2020-09-29,. MANZ SIGNS LETTER OF INTENT WITH AUTOMOTIVE ...,...,0.204794,0.800,0.854,-0.065319,0.9990,0.9965,0.002506,18.0,14.0,0.251314
2020-09-30,0.9942,0.034,0.860,0.105,75.539568,0.027870,0.413870,Positive,2020-09-30,. CAESARS ANNOUNCES RECOMMENDED CASH ACQUISITI...,...,0.105361,0.816,0.800,0.019803,0.9993,0.9990,0.000300,26.0,18.0,0.367725
2020-10-01,0.8689,0.000,0.944,0.056,100.000000,-0.016667,0.316667,Negative,2020-10-01,. SAUDI'S SIPCHEM SAYS TERMS SHEET WITH LINDE ...,...,0.125163,0.860,0.816,0.052518,0.9942,0.9993,-0.005117,16.0,26.0,-0.485508
2020-10-26,0.9866,0.030,0.890,0.080,72.727273,0.013993,0.206528,Positive,2020-10-26,. DEXCOM ANNOUNCES RETIREMENT OF RICK DOUBLEDA...,...,-inf,0.944,0.860,0.093194,0.8689,0.9942,-0.134710,5.0,16.0,-1.163151


In [ ]:
df_sentiment.loc[df_sentiment["compound"]<0.9]

In [ ]:
# df_sentiment = df_sentiment[["Date", "sentiment", "sentiment(t-1)", "sentiment(t-2)", "sentiment_return", "sentiment_ln_return", "sentiment_ewm", "PressReleaseCount"]]

In [17]:
df = df_sentiment

In [18]:
df.sort_index(inplace=True)

## VIX

In [ ]:
if (UPDATE):
    from yahoo_fin.stock_info import get_data
    df_vix = get_data("^VIX", start_date="25/09/2019", end_date=PresentDate, index_as_date = True, interval="1d")
    df_vix.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/vix.pkl")

In [19]:
df_vix=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/vix.pkl', compression=None)

df_vix.index = pd.to_datetime(df_vix.index)
df_vix["Date"]= pd.to_datetime(df_vix.index)

df_vix["vix"] = df_vix["adjclose"]
# df_vix.drop(columns=["high", "low", "adjclose", "ticker", "volume"], inplace=True)
df_vix["vix"]  = df_vix["vix"].ffill()

df_vix["vix_yesterday"] = df_vix["vix"].shift(periods=1)
# vix RETURN
df_vix["vix_return_c2c"]=(df_vix["vix"]-df_vix["vix_yesterday"])/df_vix["vix_yesterday"]
df_vix["vix_return_c2c"]=(df_vix["close"]-df_vix["open"])/df_vix["open"]
# vix LN RETURN
df_vix["vix_ln_return_c2c"]=np.log(df_vix["vix"]/df_vix["vix_yesterday"])
df_vix["vix_ln_return"]=np.log(df_vix["close"]/df_vix["open"])

# vix EWM
df_vix["vix_ewm"]=df_vix["vix"].ewm(10).mean()

In [ ]:
df_vix.tail(50)

In [ ]:
aa = df_vix.head(10)

In [20]:
df = pd.merge(df,df_vix, on='Date')
df = df.ffill()
df = df.sort_values(by="Date")

In [22]:
df.loc[df["Date"]>"2020-09-20"].head(50).sort_index()#(by=["Date"])

,compound,neg,neu,pos,sentiment,polarity,subjectivity,polarityB,Date,Articles,...,close,adjclose,volume,ticker,vix,vix_yesterday,vix_return_c2c,vix_ln_return_c2c,vix_ln_return,vix_ewm
247,0.9989,0.071,0.812,0.118,62.433862,0.092259,0.427045,Positive,2020-09-21,. WALMART SAYS TARGETS ZERO EMISSIONS BY 2040....,...,27.780001,27.780001,0,^VIX,27.780001,25.830000,-0.009272,0.072780,-0.009316,26.495012
248,0.9974,0.029,0.888,0.083,74.107143,0.023592,0.304920,Positive,2020-09-22,. DISNEY PARKS SAY HONG KONG DISNEYLAND WILL O...,...,26.860001,26.860001,0,^VIX,26.860001,27.780001,-0.061167,-0.033678,-0.063118,26.528193
249,0.9990,0.020,0.843,0.136,87.179487,0.085245,0.411495,Positive,2020-09-23,. HUAWEI SAYS STILL WILLING TO BUY CHIPS FROM ...,...,28.580000,28.580000,0,^VIX,28.580000,26.860001,0.057735,0.062069,0.056130,26.714721
250,0.9995,0.019,0.857,0.124,86.713287,0.063738,0.367166,Positive,2020-09-24,. GILEAD SCIENCES PRICES $7.25 BLN OF SENIOR U...,...,28.510000,28.510000,0,^VIX,28.510000,28.580000,-0.034868,-0.002452,-0.035490,26.877928
251,0.9965,0.022,0.854,0.123,84.827586,0.059172,0.454229,Positive,2020-09-25,. JYSELECA® (FILGOTINIB) APPROVED IN JAPAN FOR...,...,26.379999,26.379999,0,^VIX,26.379999,28.510000,-0.063543,-0.077649,-0.065651,26.832662
252,0.9990,0.027,0.800,0.173,86.500000,0.039394,0.496994,Positive,2020-09-28,. METLIFE TO REDUCE EMISSIONS BY 30 PERCENT AN...,...,26.190001,26.190001,0,^VIX,26.190001,26.379999,-0.035359,-0.007228,-0.035999,26.774238
253,0.9993,0.030,0.816,0.154,83.695652,0.058837,0.378926,Positive,2020-09-29,. MANZ SIGNS LETTER OF INTENT WITH AUTOMOTIVE ...,...,26.270000,26.270000,0,^VIX,26.270000,26.190001,-0.020142,0.003050,-0.020347,26.728398
254,0.9942,0.034,0.860,0.105,75.539568,0.027870,0.413870,Positive,2020-09-30,. CAESARS ANNOUNCES RECOMMENDED CASH ACQUISITI...,...,26.370001,26.370001,0,^VIX,26.370001,26.270000,-0.011989,0.003799,-0.012062,26.695817
255,0.8689,0.000,0.944,0.056,100.000000,-0.016667,0.316667,Negative,2020-10-01,. SAUDI'S SIPCHEM SAYS TERMS SHEET WITH LINDE ...,...,26.700001,26.700001,0,^VIX,26.700001,26.370001,0.035687,0.012437,0.035065,26.696197
256,0.9866,0.030,0.890,0.080,72.727273,0.013993,0.206528,Positive,2020-10-26,. DEXCOM ANNOUNCES RETIREMENT OF RICK DOUBLEDA...,...,32.459999,32.459999,0,^VIX,32.459999,27.549999,0.104833,0.164006,0.099694,27.875845


## S&P500

In [ ]:
if (UPDATE):
    from yahoo_fin.stock_info import get_data
    df_sp500 = get_data("^GSPC", start_date="25/09/2019", end_date=PresentDate, index_as_date = True, interval="1d")

In [ ]:
df_sp500=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sp500.pkl', compression=None)

df_sp500.index = pd.to_datetime(df_sp500.index)
df_sp500["Date"]= pd.to_datetime(df_sp500.index)

df_sp500["sp500"] = df_sp500["adjclose"]
df_sp500.drop(columns=["open", "high", "low", "close", "adjclose", "ticker", "volume"], inplace=True)
df_sp500["sp500"]  = df_sp500["sp500"].ffill()

df_sp500["sp500_yesterday"] = df_sp500["sp500"].shift(periods=1)
# sp500 RETURN
df_sp500["sp500_return"]=(df_sp500["sp500"]-df_sp500["sp500_yesterday"])/df_sp500["sp500_yesterday"]
# sp500 LN RETURN
df_sp500["sp500_ln_return"]=np.log(df_sp500["sp500"]/df_sp500["sp500_yesterday"])
# sp500 EWM
df_sp500["sp500_ewm"]=df_sp500["sp500"].ewm(10).mean()

In [ ]:
df = pd.merge(df_sp500,df,how='outer', on='Date')
df = df.ffill()
df = df.sort_values(by="Date")

In [ ]:
df

In [ ]:
# df = df.drop([475])

# DAY MONTH YEAR

In [ ]:
import datetime as dt
from datetime import date
import calendar
my_date = date.today()
calendar.day_name[my_date.weekday()]  #'Wednesday'

# Prepare data
df['year'] = [d.year for d in df.Date]
df['month'] = [d.strftime('%b') for d in df.Date]
df['day'] = [d.day_name() for d in df.Date]
years = df['year'].unique()
df['month-year'] = [d.strftime('%b') + str(d.year) for d in df.Date]

# TEXTBLOB

In [ ]:
# Reading the saved data pickle file

# df_polarity = pd.DataFrame()

# df_polarity=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentArticlesTextBlob.pkl', compression=None)
# df_polarity.drop(columns=["compound", "neg", "pos", "neu", "Articles", "Title", "Text", "Company"], inplace=True)
# df_polarity.drop(columns=["Company", "PressReleaseCount", "Articles", "Title", "Text"], inplace=True)

# df_polarity.index = pd.to_datetime(df_polarity.index)
# df_polarity.Date = pd.to_datetime(df_polarity.Date)

# df_polarity = df_polarity.ffill()

# df_polarity["subjectivity(t-1)"] = df_polarity["subjectivity"].shift(periods=1)
# df_polarity["subjectivity(t-2)"] = df_polarity["subjectivity"].shift(periods=2)

# # polarity RETURN
# df_polarity["subjectivity_return"]=(df_polarity["subjectivity(t-1)"]-df_polarity["subjectivity(t-2)"])/df_polarity["subjectivity(t-2)"]
# # polarity LN RETURN
# df_polarity["subjectivity_ln_return"]=np.log(df_polarity["subjectivity(t-1)"]/df_polarity["subjectivity(t-2)"])
# # polarity EWM
# df_polarity["subjectivity_ewm"]=df_polarity["subjectivity"].ewm(10).mean()

# df.reset_index(drop=True, inplace=True)
# df_polarity.reset_index(drop=True, inplace=True)
# df_polarity = df_polarity.ffill()

# df = pd.merge(df_polarity,df,how='outer', on='Date')
# df = df.ffill()
# df = df.sort_values(by="Date")

In [ ]:
df.index = df.Date

In [ ]:
df

# THREE PERIODS

In [ ]:
df = df.ffill()
df.index = df.Date

In [ ]:
df_part1 = df.loc['2019-09-25':'2020-02-14']
df_part2 = df.loc['2020-02-15':'2020-05-31']
df_part3 = df.loc['2020-06-01':PresentDate]
df_part3 = df_part3.ffill()

df_sentiment = df_sentiment.loc['2020-06-01':PresentDate]
df_vix = df_vix.loc['2020-06-01':PresentDate]